# Train on a free Colab GPU

Works with any dataset exported by `backend/scripts/export_for_colab.py` (run
`python scripts/export_for_colab.py <dataset_name>` from `backend/`, or use
the "Suggest cylinders"/labeling tools in the app first, then export).

**Before running:** `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.

Then: `Runtime` → `Run all`. When it gets to the upload cell, upload your `<dataset_name>-dataset.zip`.

At the end, `trained-model.zip` (containing `best.pt`) downloads automatically —
that's what you bring back to **Detect → Import a model** (desktop app or web app).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.9 MB/s eta 0:00:00


In [3]:
from google.colab import files

print("Upload your <dataset_name>-dataset.zip:")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.endswith(".zip")]
assert zip_names, "Expected a .zip file (from export_for_colab.py)"
dataset_zip = zip_names[0]
print("Using:", dataset_zip)

Upload your <dataset_name>-dataset.zip:


Saving AI_IMAGES.zip to AI_IMAGES.zip
Using: AI_IMAGES.zip


In [4]:
import shutil

shutil.unpack_archive(dataset_zip, "dataset")
!echo '--- dataset/data.yaml ---'; cat dataset/data.yaml
!echo '--- image counts ---'; find dataset/images -type f | wc -l

--- dataset/data.yaml ---
cat: dataset/data.yaml: No such file or directory
--- image counts ---
find: ‘dataset/images’: No such file or directory
0


## Train

Fine-tunes pretrained `yolov8n-seg` (matches what the desktop/web app does
locally, just on a GPU instead of CPU — labels are polygon outlines, so the
app always trains the segmentation variant, even for plain box labels).
Adjust `epochs`/`imgsz`/`batch` as needed; watch the `val` mask mAP in the
output and stop early (interrupt the cell) if it plateaus.

In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs",
    name="train",
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchs

RuntimeError: Dataset 'dataset/data.yaml' error ❌ 'dataset/data.yaml' does not exist

In [ ]:
# quick sanity check on the validation set
metrics = model.val()
print(metrics.seg.map, "(mask mAP50-95)")

In [ ]:
import shutil
from google.colab import files

shutil.copy("runs/train/weights/best.pt", "best.pt")
shutil.make_archive("trained-model", "zip", ".", "best.pt")
files.download("trained-model.zip")